# Valid claim and Invlid claim 
This program reads the oversampled training data and projects it into a new lower-dimensional space using random projection. Differential privacy is then applied by adding either Laplace or Gaussian noise to the projected data. The privacy-preserved projected data is used to train a machine learning model. Finally, the correctness of the trained model is verified using test data projected with the same random projection matrices and the same type of differential privacy noise.

In [291]:
# =====================================================
# Hyperparameters and Configuration for model training
# =====================================================

# Dataset paths
DATASET_PATH = 'Dataset/OversampledVoiceData.csv'
TESTDATASET_PATH = 'Dataset/VoiceDatatest.csv'

# Profile separation
SEPARATE_PROFILE = 68

#Train model based on plain or processed data
RPDP= True

if RPDP==False:
    N_COMPONENTS = 104
else:
   N_COMPONENTS = 94
   
# Random projection parameters
USE_RANDOM_PROJECTION = True


   
# Differential privacy parameters
USE_LAPLACE_NOISE = True
USE_GAUSSIAN_NOISE = False
EPSILON = 7.0
SENSITIVITY = 1.0
DELTA = 1e-5

# Input dimension for neural network
INPUT_DIM = N_COMPONENTS
BATCH_SIZE = 32
EPOCHS = 50

# Dynamic column generation
column1 = [f'RPF{i+1}' for i in range(N_COMPONENTS)]

# =====================================================
# Load dataset
# =====================================================

import pandas as pd
import numpy as np

dataset = pd.read_csv(DATASET_PATH, index_col=0)

In [292]:
# Separate the profiles into two groups:
# (i) training profiles (0-SEPARATE_PROFILE-1)
# (ii) auxiliary profiles (SEPARATE_PROFILE and above)

totalUser = len(pd.unique(dataset['Label']))

trainingData = dataset[dataset['Label'] < SEPARATE_PROFILE]
auxilaryData = dataset[dataset['Label'] >= SEPARATE_PROFILE]

print("Total user in training dataset:", len(pd.unique(trainingData['Label'])))
print("Total user in auxiliary dataset:", len(pd.unique(auxilaryData['Label'])))

Total user in training dataset: 68
Total user in auxiliary dataset: 18


In [293]:
#Random Project
from sklearn.random_projection import SparseRandomProjection

def apply_random_projection(data, total_user, knownR, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        if knownR==False:
            rng = np.random.RandomState(seed+np.random.randint(0, 999))
        else:    
            rng = np.random.RandomState(seed)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [294]:
# Laplace noise
def add_laplace_noise(data, total_user, epsilon=EPSILON, sensitivity=SENSITIVITY):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        scale = sensitivity / epsilon
        noise = np.random.laplace(loc=0.0, scale=scale, size=Xdata.shape)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [295]:
# Gaussioan noise
def add_gaussian_noise(data, total_user,epsilon=EPSILON,sensitivity=SENSITIVITY,delta=DELTA):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        sigma = (sensitivity * np.sqrt(2 * np.log(1.25 / delta))) / epsilon
        noise = np.random.normal(0, sigma)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [296]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = trainingData
total_user=len(pd.unique(trainingData['Label']))
# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projection(feature_data, total_user, True)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)


# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_38732\2864155557.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_38732\1620265370.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [297]:
#Prepare the traning data for training and testing the model
import tensorflow
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

if RPDP==False:
    X=trainingData.drop(columns=['Label'])
    y=trainingData['Label']
else:
    X=processed_dataset.drop(columns=['Label'])
    y=processed_dataset['Label']

Xtrain, Xval, ytrain, yval = train_test_split(X, y, test_size=0.2, random_state=22)

ytrain = to_categorical(ytrain)
yval = to_categorical(yval)
print(Xtrain.shape)
print(ytrain.shape)
print(Xval.shape)
print(yval.shape)

(10880, 94)
(10880, 68)
(2720, 94)
(2720, 68)


In [298]:
# import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#matplotlib inlineimport keras
from keras.layers import Dense, Dropout, Input,Activation,Dropout, Flatten
from keras.models import Model,Sequential
from keras.datasets import mnist

from keras.layers import BatchNormalization
from keras.optimizers import SGD, RMSprop, Adam
def adam_optimizer():
    return Adam(learning_rate=0.0002, beta_1=0.5)

def RMSprop_optimizer():
    return RMSprop(learning_rate=0.001, rho=0.9)

In [299]:
#neural network architecture

def create_classifierRP(release=False, totalClass=SEPARATE_PROFILE):
  classifier = Sequential()
  classifier.add(Dense(64, input_dim=INPUT_DIM))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.5))

  #classifier.add(Dense(256))
  #classifier.add(BatchNormalization())
  #classifier.add(Activation('relu'))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  classifier.add(Dense(256))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  #classifier.add(Dense(256))
  #classifier.add(BatchNormalization())
  #classifier.add(Activation('relu'))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  #if release:
  classifier.add(Dense(totalClass, activation='softmax'))
  #else:
  #   classifier.add(Dense(Tuser))
  #np.log_softmax_v2(a, axis=axis)
  #classifier.add(F.softmax(a, dim=1))

  classifier.compile(loss='categorical_crossentropy', optimizer=RMSprop_optimizer(),metrics=['accuracy'])
  return classifier

Clasf=create_classifierRP()
Clasf.summary()

C:\Users\mdmor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_27"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_135 (Dense)               │ (None, 64)             │         6,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_108         │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_108 (Activation)     │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_108 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_136 (Dense)               │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_109         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_109 (Activation)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_109 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_137 (Dense)               │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_110         │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_110 (Activation)     │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_110 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_138 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_111         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_111 (Activation)     │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_111 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_139 (Dense)               │ (None, 68)             │         8,772 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,396 (357.02 KB)

 Trainable params: 90,244 (352.52 KB)

 Non-trainable params: 1,152 (4.50 KB)

In [300]:
#Train the classifier
from keras.callbacks import ReduceLROnPlateau

learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', patience=5, verbose=1, factor=0.5,min_lr=0.0001)
callbacks_list = [learning_rate_reduction]

Classfier2= create_classifierRP(True,68)

#------Comment will start from here
lossc='categorical_crossentropy'
optimizerc=RMSprop(learning_rate=0.001, rho=0.9)
Classfier2.compile(loss=lossc, optimizer=optimizerc,metrics=['accuracy'])
#------Comments will end from here
historyc2 =  Classfier2.fit(Xtrain, ytrain, batch_size=64, epochs=200, validation_data=(Xval, yval),verbose=1)

Epoch 1/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1164 - loss: 3.8418 - val_accuracy: 0.8603 - val_loss: 2.4360
Epoch 2/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4906 - loss: 2.1318 - val_accuracy: 0.9757 - val_loss: 0.5244
Epoch 3/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6470 - loss: 1.3433 - val_accuracy: 0.9908 - val_loss: 0.1295
Epoch 4/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7296 - loss: 0.9904 - val_accuracy: 0.9956 - val_loss: 0.0519
Epoch 5/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7684 - loss: 0.8192 - val_accuracy: 0.9978 - val_loss: 0.0281
Epoch 6/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7941 - loss: 0.7157 - val_accuracy: 0.9982 - val_loss: 0.0177
Epoch 7/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8141 - loss: 0.6444 - val_accuracy: 0.9989 - val_loss: 0.0130
Epoch 8/200
170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8332 - loss: 0.5583 - val_accu

In [311]:
# =====================================================
# Hyperparameters and Configuration for valid and invalid claim
# =====================================================
#Type of claim
VALID_CLAIM = False
INVALID_CLAIM =True

#For invalid claim: Unknown R or known R
knownR=True

In [312]:
#read the test data either for valid or invalid claim
import csv
import pandas as pd

if VALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] < SEPARATE_PROFILE]
    
if INVALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] >= SEPARATE_PROFILE]
    newID = np.random.randint(0, SEPARATE_PROFILE, size=testdataset.shape[0])
    testdataset['Label'] = newID
    
testdataset.head()

,1,2,3,4,5,6,7,8,9,10,...,96,97,98,99,100,101,102,103,104,Label
8796,0.577301,-0.521846,-0.458939,-0.169801,0.492290,-0.608976,-0.075974,-0.057947,0.272287,-0.016334,...,0.048005,-0.225233,-0.009301,0.182901,-0.043218,-0.021152,0.009831,0.063303,-0.055561,13
8209,0.327843,-0.168374,0.086000,0.294810,-0.125812,-0.565006,0.021564,0.336860,-0.243190,-0.249655,...,-0.293992,-0.315545,0.182196,0.142055,-0.040365,-0.009444,-0.190533,0.016862,-0.503840,24
8721,0.436850,0.242402,-0.545839,0.490844,-0.176363,0.218081,-0.479460,0.485923,-0.089936,0.129415,...,0.454873,-0.007616,-0.550065,0.344938,-0.151026,0.131478,0.022711,-0.051380,0.116940,67
8875,0.824280,-0.238647,-0.423929,-0.034956,0.228715,-0.374185,-0.142289,0.114626,0.166814,0.080253,...,0.069756,-0.344372,-0.107741,0.179268,0.074635,-0.068413,-0.087623,0.231988,-0.060065,2
9294,0.537243,0.459554,0.087789,-0.018138,-0.026348,0.170799,-0.129531,-0.216600,-0.362393,0.169105,...,-0.104851,-0.217039,0.072538,0.095208,0.076355,0.045516,-0.025971,0.003157,-0.144510,37


In [308]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = testdataset
total_user=len(pd.unique(testdataset['Label']))


# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projection(feature_data, total_user, knownR)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)

# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_38732\2864155557.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_38732\1620265370.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [309]:
if RPDP==False:
    Xtest=testdataset.drop(columns=['Label'])
    ytest=testdataset['Label']
else:
    Xtest=processed_dataset.drop(columns=['Label'])
    ytest=processed_dataset['Label']
    
ytest = to_categorical(ytest)

In [310]:
#Performance of the classifier
Classfier2.compile(loss='categorical_crossentropy', optimizer=Adam(),metrics=['accuracy'])
loss, accuracy = Classfier2.evaluate(Xtest, ytest)
print('Loss:', loss)
print('Accuracy:', accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0148 - loss: 8.1825  
Loss: 8.674309730529785
Accuracy: 0.0117647061124444
